In [1]:
import pandas as pd
import numpy as np

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [2]:
# Load AID_1347034 ONLY (no AID_654 / HepG2 merge).
# Skip the 3 metadata rows (RESULT_TYPE, RESULT_DESCR, RESULT_UNIT).
aid_raw = pd.read_csv(
    '/Users/yamini/Desktop/projects/qsar casepase/qsar_prediction/Rebuttal/summary datasets/AID_1347034_datatable.csv',
    skiprows=[1, 2, 3],
    low_memory=False)

print("AID_1347034 (standalone, no HepG2 merge)")
print(f"Loaded shape: {aid_raw.shape}")
print(aid_raw['PUBCHEM_ACTIVITY_OUTCOME'].value_counts())

# Remove rows with missing CID
aid_raw['PUBCHEM_CID'] = pd.to_numeric(aid_raw['PUBCHEM_CID'], errors='coerce')
print(f"\nRows with missing CID: {aid_raw['PUBCHEM_CID'].isna().sum()}")
aid_raw = aid_raw.dropna(subset=['PUBCHEM_CID'])
aid_raw['PUBCHEM_CID'] = aid_raw['PUBCHEM_CID'].astype(int)
aid_raw['PUBCHEM_ACTIVITY_SCORE'] = pd.to_numeric(aid_raw['PUBCHEM_ACTIVITY_SCORE'], errors='coerce')

print(f"After cleanup: {aid_raw.shape}")
print(f"Unique CIDs: {aid_raw['PUBCHEM_CID'].nunique()}")

AID_1347034 (standalone, no HepG2 merge)
Loaded shape: (9667, 16)
PUBCHEM_ACTIVITY_OUTCOME
Inactive        8816
Inconclusive     436
Active           415
Name: count, dtype: int64

Rows with missing CID: 143
After cleanup: (9524, 16)
Unique CIDs: 7671


In [3]:
# Check the score -> outcome mapping in this assay so the cutoff below is justified.
# (In AID_1347034, every score >= 40 is Active, 5-30 is Inconclusive, 0 is Inactive.)
print(pd.crosstab(aid_raw['PUBCHEM_ACTIVITY_SCORE'] >= 40, aid_raw['PUBCHEM_ACTIVITY_OUTCOME']))

PUBCHEM_ACTIVITY_OUTCOME  Active  Inactive  Inconclusive
PUBCHEM_ACTIVITY_SCORE                                  
False                          0      8676           434
True                         414         0             0


In [4]:
# Merge duplicates per CID:
# 1. Average PUBCHEM_ACTIVITY_SCORE across ALL measurements for that CID.
# 2. Classify the outcome from the averaged score: 0 -> Inactive,
#    (0, 40) -> Inconclusive, >= 40 -> Active. Same rule as the merged
#    AID_654 + AID_1347034 cleaning. No 'Conflict' category.

def outcome_from_score(score):
    if score == 0:
        return 'Inactive'
    elif score < 40:
        return 'Inconclusive'
    else:
        return 'Active'

agg_dict = {col: 'first' for col in aid_raw.columns
            if col not in ['PUBCHEM_CID', 'PUBCHEM_ACTIVITY_SCORE', 'PUBCHEM_ACTIVITY_OUTCOME']}
agg_dict['PUBCHEM_ACTIVITY_SCORE'] = 'mean'

aid_clean = aid_raw.groupby('PUBCHEM_CID', as_index=False).agg(agg_dict)
aid_clean['PUBCHEM_ACTIVITY_OUTCOME'] = aid_clean['PUBCHEM_ACTIVITY_SCORE'].apply(outcome_from_score)

print(f"After dedup: {aid_raw.shape} -> {aid_clean.shape}")
print(f"Removed {len(aid_raw) - len(aid_clean)} duplicate rows")
print()
print(aid_clean['PUBCHEM_ACTIVITY_OUTCOME'].value_counts())

After dedup: (9524, 16) -> (7671, 16)
Removed 1853 duplicate rows

PUBCHEM_ACTIVITY_OUTCOME
Inactive        6968
Inconclusive     400
Active           303
Name: count, dtype: int64


In [5]:
# Verify on a CID whose measurements disagreed on outcome (if any).
n_outcomes_per_cid = aid_raw.groupby('PUBCHEM_CID')['PUBCHEM_ACTIVITY_OUTCOME'].nunique()
disagreed_cids = n_outcomes_per_cid[n_outcomes_per_cid > 1].index
print(f"Total CIDs whose measurements disagreed on outcome: {len(disagreed_cids)}")

if len(disagreed_cids) > 0:
    example_cid = disagreed_cids[0]
    cols = ['PUBCHEM_CID', 'PUBCHEM_ACTIVITY_OUTCOME', 'PUBCHEM_ACTIVITY_SCORE']
    print(f"\nExample CID {example_cid} - before dedup:")
    display(aid_raw[aid_raw['PUBCHEM_CID'] == example_cid][cols])
    print("After dedup (score averaged, outcome from averaged score):")
    display(aid_clean[aid_clean['PUBCHEM_CID'] == example_cid][cols])

Total CIDs whose measurements disagreed on outcome: 125

Example CID 727 - before dedup:


,PUBCHEM_CID,PUBCHEM_ACTIVITY_OUTCOME,PUBCHEM_ACTIVITY_SCORE
421,727,Inconclusive,30
515,727,Inconclusive,30
3520,727,Inactive,0
3806,727,Inactive,0
6918,727,Inactive,0
8179,727,Inactive,0
8378,727,Inactive,0
9524,727,Inactive,0


After dedup (score averaged, outcome from averaged score):


,PUBCHEM_CID,PUBCHEM_ACTIVITY_OUTCOME,PUBCHEM_ACTIVITY_SCORE
72,727,Inconclusive,7.5


In [6]:
# Save
aid_clean.to_csv('AID_1347034_cleaned.csv', index=False)
print(f"Saved {len(aid_clean)} compounds to AID_1347034_cleaned.csv")

Saved 7671 compounds to AID_1347034_cleaned.csv
